# Colab Setup

In [ ]:
import sys
import os
import torch
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, HTML

BASE_DIR = "/content/multilingual_bias"

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    if os.path.isdir(BASE_DIR):
        import shutil
        print("Removing existing repo to update")
        shutil.rmtree(BASE_DIR)
  
    from git import Repo
    
    os.chdir("/")

    repo_url = "https://github.com/jonesmonez/multilingual_bias"
    Repo.clone_from(repo_url, BASE_DIR, branch="python312", single_branch=True)
    
    os.chdir(BASE_DIR)
    print(f"Repository cloned to {BASE_DIR}")

Repository cloned to /content/multilingual_bias


In [ ]:
load_dotenv()

value = os.getenv('KAGGLE_API_TOKEN')

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
api.dataset_download_files('dudewithpants/wiki-dump', path='.', unzip=True)   

import kaggle

# Crows Tests

In [1]:
from experiments.modules.crows_runner import CrowSPairsRunnerWrapper

runner = CrowSPairsRunnerWrapper(
    save_result = True
)

# to_test_types = ["gender", "race-color", "religion"]
to_test_types = ["religion"]
# to_test_lang = ["ar_DZ", "ca_ES", "de_DE", "en_US", "es_AR", "fr_FR", "it_IT", "mt_MT", "zh_CN"]
to_test_lang = ["mt_MT"]
debias_lang = ["mt_MT"]

In [2]:
import pandas as pd
results_base = {}

for lang in to_test_lang:
    result_btype = {}
    
    for btype in to_test_types:
        
        result = runner.run_base(
            path_to_crows=f"data/crows_improved/crows_{lang}.csv",
            lang_eval=lang,
            bias_type=btype,
        )
        
        result_btype[btype] = result[0]
    
    results_base[lang] = result_btype
    
print(results_base)
display(pd.DataFrame.from_dict(results_base, orient="index"))

  0%|          | 0/114 [00:00<?, ?it/s]

{'mt_MT': {'religion': 52.63}}


,religion
mt_MT,52.63


In [ ]:
import torch
results_debias = {}

for dlang in debias_lang:
    result_lang = {}
    
    for lang in to_test_lang:
        result_btype = {}
        
        for btype in to_test_types:
            
            if btype == "race-color":
                btype_path = "racecolor"
            else:
                btype_path = btype
            
            result = runner.run_debias(
                path_to_crows=f"data/crows_improved/crows_{lang}.csv",
                lang_debias=dlang,
                lang_eval=lang,
                bias_type=btype,
                bias_direction=f"data/subspace/{btype_path}_subspace_{dlang}.pt",
            )
            
            result_btype[btype] = result[0]
            
        torch.cuda.empty_cache()
        
        result_lang[lang] = result_btype
    
    results_debias[dlang] = result_lang
    
    print(result_lang)
    display(pd.DataFrame.from_dict(result_lang, orient="index"))
    
print(results_debias)

  0%|          | 0/114 [00:00<?, ?it/s]

NameError: name 'torch' is not defined

# CDA & Dropout

In [2]:
from experiments.modules.debias_trainer import DropoutTrainer, CDATrainer

ModuleNotFoundError: No module named 'experiments'

In [3]:
dropout_trainer = DropoutTrainer(
    model_name_or_path="bert-base-multilingual-uncased",
    max_seq_length=128,
    seed=42,
    fp16=True,
)

In [14]:
dropout_model_dir = dropout_trainer.train(
    train_file="wiki_de_DE.txt",
    output_dir="./debias_models/bert-base-uncased_dropout",
    num_train_epochs=3,
    per_device_train_batch_size=24,
    learning_rate=3e-5,
    logging_steps=200,
    save_steps=500,
    warmup_steps=500,
    max_grad_norm=1.0,
)

In [5]:
print(dropout_model_dir)

debias_models/bert-base-uncased_dropout


# INLP

In [1]:
from experiments.modules.inlp_runner import InlpRunner
runner = InlpRunner(
    "BertModel",
    "bert-base-multilingual-uncased",
)
import nltk
nltk.download('punkt_tab')
runner.setup_data(
    "wiki_de_DE.txt",
    "data/bias_attribute/translated/bias_attribute_words_de_DE.json",
    "de_DE",
    "religion"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[nltk_data] Downloading package punkt_tab to /home/jonas/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


FileNotFoundError: [Errno 2] No such file or directory: 'wiki_de_DE.txt'

In [10]:
runner.compute_projection_matrix()

Encoding neutral sentences: 100%|██████████| 10000/10000 [02:10<00:00, 76.40it/s]


Dataset split sizes:
Train size: 9800; Dev size: 4200; Test size: 6000


iteration: 79, accuracy: 0.5573809523809524: 100%|██████████| 80/80 [07:29<00:00,  5.62s/it]


tensor([[ 8.8124e-01, -2.2041e-04, -8.0665e-03,  ..., -3.9003e-03,
          4.7755e-04,  2.6445e-03],
        [-2.2041e-04,  8.9042e-01,  2.7494e-02,  ..., -1.1926e-02,
          9.7388e-04,  9.2103e-05],
        [-8.0665e-03,  2.7494e-02,  8.3932e-01,  ..., -4.9165e-02,
         -4.1225e-03, -7.6228e-03],
        ...,
        [-3.9003e-03, -1.1926e-02, -4.9165e-02,  ...,  8.8629e-01,
         -7.1935e-03, -5.8769e-03],
        [ 4.7755e-04,  9.7388e-04, -4.1225e-03,  ..., -7.1935e-03,
          9.2841e-01,  4.9368e-03],
        [ 2.6445e-03,  9.2103e-05, -7.6228e-03,  ..., -5.8769e-03,
          4.9368e-03,  9.2423e-01]])

In [24]:
import os
from pathlib import Path

class TestClass:
    def __init__(
        self,
        some_path: Path = Path("test")
    ):        
        some_path = some_path / ""
        print(f"Der Path ist: {some_path.resolve()}")
        
        
        some_path.mkdir(parents=True, exist_ok=True)
        
        
t = TestClass()

Der Path ist: /home/jonas/code/multilingual_bias/test
